# Load both datasets

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [6]:
from google.colab import files
uploaded = files.upload()

Saving E0.xlsx to E0 (1).xlsx


In [7]:
epl = pd.read_excel("E0.xlsx")


In [8]:
from google.colab import files
uploaded = files.upload()

Saving D1.xlsx to D1.xlsx


In [10]:
bundesliga = pd.read_excel("D1.xlsx")

In [11]:
print(epl.shape)
print(bundesliga.shape)

(349, 132)
(288, 131)


# Add league labels

In [12]:
epl["League"] = "Premier League"
bundesliga["League"] = "Bundesliga"

# Combine datasets

In [13]:
combined = pd.concat(
    [epl, bundesliga],
    ignore_index=True
)

In [14]:
combined.head()

,Div,Date,Time,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,...,B365CAHA,PCAHH,PCAHA,MaxCAHH,MaxCAHA,AvgCAHH,AvgCAHA,BFECAHH,BFECAHA,League
0,E0,2025-08-15,20:00:00,Liverpool,Bournemouth,4,2,H,1,0,...,1.78,2.07,1.85,2.03,1.88,1.94,1.76,2.14,1.86,Premier League
1,E0,2025-08-16,12:30:00,Aston Villa,Newcastle,0,0,D,0,0,...,1.80,2.02,1.89,2.06,1.80,1.95,1.74,2.14,1.86,Premier League
2,E0,2025-08-16,15:00:00,Brighton,Fulham,1,1,D,0,0,...,2.03,1.93,2.00,1.84,2.03,1.80,1.96,1.91,2.08,Premier League
3,E0,2025-08-16,15:00:00,Sunderland,West Ham,3,0,H,0,0,...,1.90,1.97,1.95,1.95,1.94,1.86,1.78,2.02,1.97,Premier League
4,E0,2025-08-16,15:00:00,Tottenham,Burnley,3,0,H,1,0,...,1.88,1.99,1.93,1.98,1.91,1.88,1.83,2.07,1.92,Premier League


In [15]:
combined["League"].value_counts()

,count
League,
Premier League,349
Bundesliga,288


# Convert odds into probabilities

In [16]:
combined["home_prob"] = 1 / combined["B365H"]
combined["draw_prob"] = 1 / combined["B365D"]
combined["away_prob"] = 1 / combined["B365A"]

# Normalize probabilities

In [17]:
total = (
    combined["home_prob"] +
    combined["draw_prob"] +
    combined["away_prob"]
)

combined["home_prob"] /= total
combined["draw_prob"] /= total
combined["away_prob"] /= total

# Encode actual match outcomes

In [18]:
combined["actual_home"] = (
    combined["FTR"] == "H"
).astype(int)

combined["actual_draw"] = (
    combined["FTR"] == "D"
).astype(int)

combined["actual_away"] = (
    combined["FTR"] == "A"
).astype(int)

# Calculate prediction error

In [19]:
combined["home_error"] = abs(
    combined["actual_home"] -
    combined["home_prob"]
)

combined["draw_error"] = abs(
    combined["actual_draw"] -
    combined["draw_prob"]
)

combined["away_error"] = abs(
    combined["actual_away"] -
    combined["away_prob"]
)

# Calculate league-level MAE

In [20]:
league_mae = combined.groupby("League")[[
    "home_error",
    "draw_error",
    "away_error"
]].mean()

league_mae["overall_mae"] = (
    league_mae["home_error"] +
    league_mae["draw_error"] +
    league_mae["away_error"]
) / 3

league_mae

,home_error,draw_error,away_error,overall_mae
League,,,,
Bundesliga,0.413812,0.364061,0.375010,0.384294
Premier League,0.435544,0.379269,0.388888,0.401234


In [21]:
import plotly.express as px

fig = px.bar(
    league_mae.reset_index(),
    x="League",
    y="overall_mae",
    color="League",
    text="overall_mae",
    title="Bookmaker Prediction Error by League"
)

fig.show()

# “Are bookmakers better at pricing favorites or outsiders?

# Create market categories

In [22]:
def classify_market(odds):

    if odds < 2.0:
        return "Favorite"

    elif odds <= 3.5:
        return "Balanced"

    else:
        return "Underdog"

In [23]:
combined["market_type"] = combined["B365H"].apply(
    classify_market
)

# Measure prediction accuracy

In [24]:
market_analysis = combined.groupby(
    ["League", "market_type"]
)[["home_error"]].mean().reset_index()

market_analysis

,League,market_type,home_error
0,Bundesliga,Balanced,0.438722
1,Bundesliga,Favorite,0.442526
2,Bundesliga,Underdog,0.243264
3,Premier League,Balanced,0.464403
4,Premier League,Favorite,0.465472
5,Premier League,Underdog,0.304611


In [25]:
import plotly.express as px

fig = px.bar(
    market_analysis,
    x="market_type",
    y="home_error",
    color="League",
    barmode="group",
    text="home_error",
    title="Bookmaker Prediction Error: Favorites vs Underdogs"
)

fig.show()

In [26]:
import plotly.express as px

league_mae = league_mae.reset_index()

fig = px.bar(
    league_mae,
    x="League",
    y="overall_mae",
    color="League",
    text="overall_mae",
    title="Bookmaker Prediction Accuracy by League"
)

fig.update_traces(
    texttemplate='%{text:.3f}',
    textposition='outside'
)

fig.show()

# When bookmakers imply a 70% chance, does the team actually win ~70% of the time?”
Do bookmaker probabilities match real-world outcomes?

# Create probability bins

In [27]:
combined["prob_bin"] = pd.cut(
    combined["home_prob"],
    bins=[0,0.1,0.2,0.3,0.4,0.5,
          0.6,0.7,0.8,0.9,1.0]
)

# Calculate actual win rates

In [28]:
calibration = combined.groupby(
    ["League", "prob_bin"]
).agg(
    expected_prob=("home_prob", "mean"),
    actual_win_rate=("actual_home", "mean"),
    matches=("actual_home", "count")
).reset_index()

/tmp/ipykernel_17079/181324851.py:1: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



In [29]:
calibration.head(15)

,League,prob_bin,expected_prob,actual_win_rate,matches
0,Bundesliga,"(0.0, 0.1]",0.082415,0.000000,4
1,Bundesliga,"(0.1, 0.2]",0.156367,0.000000,15
2,Bundesliga,"(0.2, 0.3]",0.259851,0.162162,37
3,Bundesliga,"(0.3, 0.4]",0.353566,0.233333,60
4,Bundesliga,"(0.4, 0.5]",0.450680,0.441558,77
5,Bundesliga,"(0.5, 0.6]",0.547298,0.707317,41
6,Bundesliga,"(0.6, 0.7]",0.643604,0.729730,37
7,Bundesliga,"(0.7, 0.8]",0.752772,0.875000,8
8,Bundesliga,"(0.8, 0.9]",0.850185,0.777778,9
9,Bundesliga,"(0.9, 1.0]",NaN,NaN,0


In [30]:
import plotly.express as px

fig = px.line(
    calibration,
    x="expected_prob",
    y="actual_win_rate",
    color="League",
    markers=True,
    title="Probability Calibration: Expected vs Actual Outcomes"
)

fig.add_scatter(
    x=[0,1],
    y=[0,1],
    mode="lines",
    name="Perfect Calibration"
)

fig.show()

# Favorite Inflation Analysis

# Business Question

“Do bookmakers systematically overestimate strong favorites?”

# Define strong favorites

In [31]:
strong_favorites = combined[
    combined["home_prob"] >= 0.7
]

# Compare expectation vs reality

In [32]:
favorite_analysis = strong_favorites.groupby(
    "League"
).agg(
    expected_prob=("home_prob", "mean"),
    actual_win_rate=("actual_home", "mean"),
    matches=("actual_home", "count")
).reset_index()

favorite_analysis

,League,expected_prob,actual_win_rate,matches
0,Bundesliga,0.804344,0.823529,17
1,Premier League,0.764429,0.823529,17


In [33]:
import plotly.graph_objects as go

fig = go.Figure()

fig.add_trace(go.Bar(
    x=favorite_analysis["League"],
    y=favorite_analysis["expected_prob"],
    name="Expected Probability"
))

fig.add_trace(go.Bar(
    x=favorite_analysis["League"],
    y=favorite_analysis["actual_win_rate"],
    name="Actual Win Rate"
))

fig.update_layout(
    barmode="group",
    title="Favorite Inflation Analysis",
    yaxis_title="Probability / Win Rate"
)

fig.show()